In [ ]:
"""
One person tells a neighbor (local improvement)
That neighbor tells another
Eventually the whole network knows (global convergence)

Bellman-Ford works because repeated local improvements propagate information through the graph until the full global picture emerges.
"""

In [1]:
import math

def initialize_single_source(vertices, source):
    dist = {v: math.inf for v in vertices}
    pred = {v: None for v in vertices}
    dist[source] = 0
    return dist, pred

def relax(u, v, weight, dist, pred):
    if dist[u] + weight < dist[v]:
        dist[v] = dist[u] + weight
        pred[v] = u

def bellman_ford(vertices, edges, source):
    dist, pred = initialize_single_source(vertices, source)

    for _ in range(len(vertices) - 1):
        for (u, v, w) in edges:
            relax(u, v, w, dist, pred)

    # negative-weight cycles
    for (u, v, w) in edges:
        if dist[u] + w < dist[v]:
            return False, None, None

    return True, dist, pred

In [2]:
vertices = ['A', 'B', 'C', 'D']
edges = [
    ('A', 'B', 1),
    ('B', 'C', 3),
    ('A', 'C', 10),
    ('C', 'D', -2),
    ('D', 'B', -1)
]

ok, dist, pred = bellman_ford(vertices, edges, 'A')

if ok:
    print("Distances:", dist)
    print("Predecessors:", pred)
else:
    print("Graph contains a negative-weight cycle")

Distances: {'A': 0, 'B': 1, 'C': 4, 'D': 2}
Predecessors: {'A': None, 'B': 'A', 'C': 'B', 'D': 'C'}


In [6]:
"""
Modify Bellman-Ford to terminate early if a full pass over all edges performs no relaxations.
Since shortest paths have at most m edges, the algorithm will stop after at most m+1 passes.
"""
import math

def bellman_ford(vertices, edges, source):
    dist = {v: math.inf for v in vertices}
    pred = {v: None for v in vertices}
    dist[source] = 0

    for _ in range(len(vertices) - 1):
        updated = False

        for (u, v, w) in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u
                updated = True

        if not updated:
            break   # EARLY TERMINATION
            
    return dist, pred

In [7]:
"""
After the standard Bellman-Ford relaxation, perform an additional pass
to identify vertices whose distances can still be improved, mark them as part
of or affected by a negative cycle, and propagate −∞ from these vertices to all
reachable vertices.
"""
def bellman_ford(vertices, edges, source):
    dist = {v: float('inf') for v in vertices}
    pred = {v: None for v in vertices}

    dist[source] = 0

    for _ in range(len(vertices) - 1):
        for (u, v, w) in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u

    affected = set()

    for (u, v, w) in edges:
        if dist[u] + w < dist[v]:
            affected.add(v)

    from collections import deque

    queue = deque(affected)

    while queue:
        u = queue.popleft()

        if dist[u] != float('-inf'):
            dist[u] = float('-inf')

        for (a, b, w) in edges:
            if a == u and dist[b] != float('-inf'):
                dist[b] = float('-inf')
                queue.append(b)

    return dist, pred

In [8]:
"""
Bellman-Ford normally computes:
shortest paths FROM one source s

But here we want:
shortest paths TO v from ANY source

Instead of trying every possible source, we simulate "all sources at once"
by adding a fake node that connects to everything for free.

Add a new source vertex s* and connect it to every vertex v ∈ V with an edge of weight 0.
Then run Bellman-Ford with s* as the source. The resulting distances d[v] equal δ*(v) for all v.
The running time is O(VE).
"""
def compute_delta_star(vertices, edges):
    s_star = "super_source"

    new_edges = edges.copy()
    for v in vertices:
        new_edges.append((s_star, v, 0))

    dist = {v: float('inf') for v in vertices}
    dist[s_star] = 0

    for _ in range(len(vertices)):
        for (u, v, w) in new_edges:
            if dist.get(u, float('inf')) + w < dist.get(v, float('inf')):
                dist[v] = dist[u] + w

    return {v: dist[v] for v in vertices}

In [10]:
"""
The algorithm does not return to the source
It returns a closed loop of vertices forming a negative cycle

The loop is found by:
detecting a bad vertex
walking into the cycle
looping until you return to the start

You follow predecessor pointers until they repeat because repetition means you've found a cycle.
"""
def find_negative_cycle(vertices, edges, source):
    dist = {v: float('inf') for v in vertices}
    pred = {v: None for v in vertices}
    dist[source] = 0

    for _ in range(len(vertices) - 1):
        for (u, v, w) in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                pred[v] = u

    # negative cycle
    x = None
    for (u, v, w) in edges:
        if dist[u] + w < dist[v]:
            x = v
            break

    if x is None:
        return None  # No negative cycle

    for _ in range(len(vertices)):
        x = pred[x]

    cycle = []
    start = x

    while True:
        cycle.append(x)
        x = pred[x]
        if x == start:
            cycle.append(x)
            break

    cycle.reverse()
    return cycle

In [11]:
def topological_sort(G):
    visited = set()
    stack = []

    def dfs(u):
        visited.add(u)
        for v in G[u]:
            if v not in visited:
                dfs(v)
        stack.append(u)

    for u in G:
        if u not in visited:
            dfs(u)

    return stack[::-1]  # reverse to get correct order

In [12]:
def dag_shortest_paths(G, w, s):
    topo_order = topological_sort(G)
    
    d = {v: float('inf') for v in G}
    pi = {v: None for v in G}
    d[s] = 0
    
    for u in topo_order:
        for v in G[u]:
            if d[v] > d[u] + w[(u, v)]:
                d[v] = d[u] + w[(u, v)]
                pi[v] = u
    
    return d, pi

In [13]:
G = {
    'r': ['s', 't'],
    's': ['t', 'x'],
    't': ['x', 'y', 'z'],
    'x': ['y', 'z'],
    'y': ['z'],
    'z': []
}

w = {
    ('r','s'): 5, ('r','t'): 3,
    ('s','t'): 2, ('s','x'): 6,
    ('t','x'): 7, ('t','y'): 4, ('t','z'): 2,
    ('x','y'): -1, ('x','z'): 1,
    ('y','z'): -2
}

d, pi = dag_shortest_paths(G, w, 'r')

print("Distances:", d)
print("Predecessors:", pi)

Distances: {'r': 0, 's': 5, 't': 3, 'x': 10, 'y': 7, 'z': 5}
Predecessors: {'r': None, 's': 'r', 't': 'r', 'x': 't', 'y': 't', 'z': 't'}


In [15]:
"""
Tasks = vertices
Dependencies = edges
Output = minimum time to finish the project

We flipped shortest-path into longest-path to find the bottleneck chain of tasks,
which determines total completion time.
"""
def dag_longest_paths(G, weight, s):
    topo_order = topological_sort(G)
    
    d = {v: float('-inf') for v in G}
    pi = {v: None for v in G}
    d[s] = weight[s]
    
    for u in topo_order:
        for v in G[u]:
            if d[v] < d[u] + weight[v]:   # MAX instead of MIN
                d[v] = d[u] + weight[v]
                pi[v] = u
    
    return d, pi

In [16]:
G = {
    'A': ['B', 'C'],
    'B': ['D'],
    'C': ['D'],
    'D': ['E'],
    'E': []
}

weight = {
    'A': 3,
    'B': 2,
    'C': 4,
    'D': 2,
    'E': 1
}

source = 'A'

print(dag_longest_paths(G, weight, source))

({'A': 3, 'B': 5, 'C': 7, 'D': 9, 'E': 10}, {'A': None, 'B': 'A', 'C': 'A', 'D': 'C', 'E': 'D'})
